In [1]:
# Import the tools for data loading and model comparison.
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from pycaret.classification import ClassificationExperiment

print("Ready to compare classification models.")

Ready to compare classification models.


In [2]:
# Locate the dataset relative to the notebooks folder.
data_path = Path("../data/raw/diabetic_data.csv")

# Treat "?" as missing, while preserving the literal category "None".
df = pd.read_csv(
    data_path,
    keep_default_na=False,
    na_values=["?"]
)

# Define the target: 1 = readmitted within 30 days; 0 = otherwise.
df["early_readmission"] = df["readmitted"].map({
    "<30": 1,
    ">30": 0,
    "NO": 0
})

# Stop if any readmission category could not be mapped.
assert df["early_readmission"].notna().all(), "Unexpected readmission category."

print("Dataset shape:", df.shape)
print("\nTarget counts:")
print(df["early_readmission"].value_counts().sort_index())

Dataset shape: (101766, 51)

Target counts:
early_readmission
0    90409
1    11357
Name: count, dtype: int64


In [3]:
# Split by patient to keep each person's visits in one partition.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df["early_readmission"],
        groups=df["patient_nbr"]
    )
)

train_data = df.iloc[train_idx].copy()
test_data = df.iloc[test_idx].copy()

# Verify that no patient appears in both partitions.
shared_patients = set(train_data["patient_nbr"]) & set(
    test_data["patient_nbr"]
)
assert len(shared_patients) == 0, "Patient overlap detected."

print("Training encounters:", len(train_data))
print("Test encounters:", len(test_data))
print("Shared patients:", len(shared_patients))

Training encounters: 81613
Test encounters: 20153
Shared patients: 0


In [4]:
# Keep patient IDs separately for grouped cross-validation.
train_groups = train_data["patient_nbr"].reset_index(drop=True)

# Remove identifiers, the original outcome, and mostly missing weight.
excluded_columns = [
    "encounter_id",
    "patient_nbr",
    "readmitted",
    "weight"
]

train_model = train_data.drop(columns=excluded_columns).reset_index(drop=True)
test_model = test_data.drop(columns=excluded_columns).reset_index(drop=True)

# Identify constant predictors using training data only.
constant_columns = [
    column
    for column in train_model.columns
    if column != "early_readmission"
    and train_model[column].nunique(dropna=False) <= 1
]

# Apply the same column removal to both partitions.
train_model = train_model.drop(columns=constant_columns)
test_model = test_model.drop(columns=constant_columns)

print("Constant columns removed:", constant_columns)
print("Predictors:", train_model.shape[1] - 1)

Constant columns removed: ['acetohexamide', 'examide', 'citoglipton', 'metformin-pioglitazone']
Predictors: 42


In [5]:
# These variables represent numerical quantities.
numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

# All remaining predictors represent categories.
categorical_features = [
    column
    for column in train_model.columns
    if column not in numeric_features + ["early_readmission"]
]

print("Numerical predictors:", len(numeric_features))
print("Categorical predictors:", len(categorical_features))

Numerical predictors: 8
Categorical predictors: 34


In [10]:
# Create a classification experiment.
experiment = ClassificationExperiment()

# Configure preprocessing and patient-grouped cross-validation.
experiment.setup(
    data=train_model,
    test_data=test_model,
        index=False,
    target="early_readmission",
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    numeric_imputation="median",
    categorical_imputation="constant",
    normalize=True,
    fold_strategy="groupkfold",
    fold_groups=train_groups,
    fold=3,
    session_id=42,
    n_jobs=1,
    verbose=True
)

,Description,Value
0,Session id,42
1,Target,early_readmission
2,Target type,Binary
3,Original data shape,"(101766, 43)"
4,Transformed data shape,"(101766, 143)"
5,Transformed train set shape,"(81613, 143)"
6,Transformed test set shape,"(20153, 143)"
7,Numeric features,8
8,Categorical features,34
9,Rows with missing values,73.7%


In [11]:
# Compare five models and select the highest cross-validation AUC.
best_model = experiment.compare_models(
    include=["lr", "dt", "rf", "et", "lightgbm"],
    sort="AUC",
    n_select=1,
    errors="raise"
)

# Save and display the comparison results.
comparison_results = experiment.pull()
display(comparison_results)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.8870,0.6696,0.0201,0.4797,0.0386,0.0298,0.0798,3.4667
lr,Logistic Regression,0.8864,0.6651,0.0274,0.4486,0.0516,0.0390,0.0879,3.3300
rf,Random Forest Classifier,0.8873,0.6456,0.0021,0.5940,0.0041,0.0033,0.0299,8.9100
et,Extra Trees Classifier,0.8870,0.6308,0.0027,0.3813,0.0054,0.0039,0.0242,10.1133
dt,Decision Tree Classifier,0.8002,0.5309,0.1832,0.1610,0.1714,0.0583,0.0585,3.2167


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.8870,0.6696,0.0201,0.4797,0.0386,0.0298,0.0798,3.4667
lr,Logistic Regression,0.8864,0.6651,0.0274,0.4486,0.0516,0.0390,0.0879,3.3300
rf,Random Forest Classifier,0.8873,0.6456,0.0021,0.5940,0.0041,0.0033,0.0299,8.9100
et,Extra Trees Classifier,0.8870,0.6308,0.0027,0.3813,0.0054,0.0039,0.0242,10.1133
dt,Decision Tree Classifier,0.8002,0.5309,0.1832,0.1610,0.1714,0.0583,0.0585,3.2167
